# hEDS: SNP and indel pilot EDA
Run inside All of Us only. Start with the phenotype discovery cell; **no cohorts exist yet**.
This notebook cannot identify all clinically diagnosed hEDS participants automatically.
Generic EDS, hypermobility, and absent diagnosis codes are not confirmed hEDS or healthy labels.

Use **Python (hEDSpace)** for concept discovery. Genomic cells require a supported Hail/Spark app and a **dense, split, biallelic, QC-reviewed GRCh38 MatrixTable** from the same CDR release. Do not feed sparse VDS variant data into this notebook: missing entries are not reference genotypes. Do not install Hail into the ordinary Jupyter app and assume it has Spark configured.

By default all autosomal sites in the supplied callset are analyzed. For a cheap smoke test, set INTERVAL to a single region first. The selected callset defines genomic coverage; a smaller callset is not the whole genome. The configured 500:1 sampling ratio is a study design choice, not a prevalence estimate. No association, causality, or interaction tests are performed.


In [ ]:
# Configuration: paths refer to files INSIDE Workbench.
PROJECT = "wb-spotless-pecan-6923"
CDR = "wb-silky-artichoke-2408.C2025Q4R6"
COHORT_TSV = ""  # reviewed sample_id,group manifest (case or control); see below
MT_PATH = ""     # confirmed CDRv9 dense Hail MatrixTable gs://.../*.mt
INTERVAL = None  # all autosomes; use "chr1:1000000-2000000" for a 1 Mb smoke test
CONTROLS_PER_CASE = 500
SEED = 20261111
MIN_CALL_RATE = 0.98
PHENOTYPE_REVIEWED = False
CALLSET_QC_REVIEWED = False


## 1. Discover phenotype vocabulary (run now)
This queries vocabulary only, not participant records. Review both source and standard concepts and their mappings with a phenotype expert. ICD-10-CM Q79.62 may be a useful candidate, but code presence does not confirm the clinical diagnostic criteria; older generic EDS codes cannot distinguish subtypes. Include concept history, timing, exclusion diagnoses, and sufficient EHR follow-up in the eventual phenotype.


In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)
concepts = client.query(f"""
SELECT concept_id, concept_name, vocabulary_id, concept_code,
       standard_concept, invalid_reason
FROM `{CDR}.concept`
WHERE REGEXP_CONTAINS(LOWER(concept_name), r'ehlers|hypermobil')
   OR (vocabulary_id = 'ICD10CM' AND concept_code LIKE 'Q79.6%')
ORDER BY vocabulary_id, concept_code
""", job_config=bigquery.QueryJobConfig(maximum_bytes_billed=2_000_000_000)).to_dataframe()
display(concepts)  # vocabulary metadata only


## 2. Supply a reviewed, genomics-eligible cohort manifest
Create a TSV **inside Workbench** with exactly `sample_id` and `group` columns, one unrelated, QC-passing WGS sample per participant. `sample_id` must match the MatrixTable's `s` field using the release's documented participant/sample mapping. Groups must be `case` or `control`.

Include all eligible reviewed hEDS cases and the full eligible control pool, not a preselected handful. Controls mean *no recorded phenotype under your reviewed exclusion definition*, not healthy people. Exclude EDS/hypermobility and competing connective-tissue diagnoses as appropriate; review age, EHR observation, ancestry, sex, relatedness and sequencing batch. Do not exclude ordinary comorbidities without a prespecified rationale. One sample per participant, case/control disjointness and sample-ID mapping must be verified before setting the review flags.

There is intentionally no automatic cohort definition while these decisions are unresolved. After review, set the manifest/callset paths and flags. Sampling below happens only after intersecting with the available callset. Aggregate plots are descriptive and unadjusted; use ancestry/batch-stratified sensitivity checks before interpreting differences. Do not copy participant IDs, genotypes, or small-cell outputs out of Workbench.


In [ ]:
import hashlib
import pandas as pd
assert PHENOTYPE_REVIEWED and CALLSET_QC_REVIEWED, "Review phenotype, sample mapping, unrelatedness and release QC first."
assert COHORT_TSV and MT_PATH, "Supply the reviewed cloud manifest and dense MatrixTable path."
cohort = pd.read_csv(COHORT_TSV, sep="\t", dtype=str)
assert set(cohort.columns) == {"sample_id", "group"}
assert not cohort.isna().any().any() and cohort.sample_id.is_unique
assert set(cohort.group) == {"case", "control"}
assert cohort.sample_id.str.len().gt(0).all()

import hail as hl  # run this and later cells in a supported Hail/Spark environment
hl.init(default_reference="GRCh38")
mt = hl.read_matrix_table(MT_PATH)
assert list(mt.col_key) == ["s"] and mt.s.dtype == hl.tstr
assert mt.locus.dtype.reference_genome.name == "GRCh38"
assert "GT" in mt.entry and "filters" in mt.row, "Expected documented GT and variant filters fields."
available = mt.cols().select().to_pandas()["s"]
# IDs stay in cloud memory; never display them.
cohort = cohort[cohort.sample_id.isin(available)].copy()
cases = cohort[cohort.group.eq("case")]
controls = cohort[cohort.group.eq("control")].copy()
n_case = len(cases)
assert n_case >= 20, "Insufficient reviewed cases for this pilot's aggregate display threshold."
target = n_case * CONTROLS_PER_CASE
# Stable hash ranking avoids dependence on file row order.
controls["rank"] = controls.sample_id.map(lambda s: hashlib.sha256(f"{SEED}:{s}".encode()).hexdigest())
controls = controls.sort_values(["rank", "sample_id"]).head(target).drop(columns="rank")
assert len(controls) >= 20, "Insufficient eligible controls."
print({"eligible_cases": n_case, "controls_requested": target,
       "controls_selected": len(controls), "shortfall": max(0, target-len(controls)),
       "actual_controls_per_case": len(controls)/n_case})
selected = pd.concat([cases, controls], ignore_index=True)
labels = hl.Table.from_pandas(selected).key_by("sample_id")
mt = mt.annotate_cols(group=labels[mt.s].group)
mt = mt.filter_cols(hl.is_defined(mt.group))


## 3. Compare SNPs and indels
Uses PASS sites, biallelic SNPs/indels, autosomes and the same sites in both groups.
The release/sample/entry QC must already have been applied upstream. A 98% call-rate filter is then applied **in each group separately** to limit differential missingness. Missing genotypes remain missing; allele frequencies use called allele counts. Haploid/non-diploid genotypes are masked for this autosomal pilot.


In [ ]:
if INTERVAL is not None:
    mt = hl.filter_intervals(mt, [hl.parse_locus_interval(INTERVAL, reference_genome="GRCh38")])
mt = mt.filter_rows(mt.locus.in_autosome() & (hl.len(mt.alleles) == 2))
mt = mt.filter_rows(hl.is_defined(mt.filters) & (hl.len(mt.filters) == 0))
mt = mt.filter_rows(hl.is_snp(mt.alleles[0], mt.alleles[1]) | hl.is_indel(mt.alleles[0], mt.alleles[1]))
mt = mt.annotate_rows(kind=hl.if_else(hl.is_snp(mt.alleles[0], mt.alleles[1]), "SNP", "indel"))
mt = mt.annotate_entries(GT=hl.or_missing(mt.GT.ploidy == 2, mt.GT))
def group_stats(group):
    return hl.agg.filter(mt.group == group, hl.struct(
        call_rate=hl.agg.fraction(hl.is_defined(mt.GT)),
        AC=hl.agg.sum(hl.or_missing(hl.is_defined(mt.GT), mt.GT.n_alt_alleles())),
        AN=hl.agg.sum(hl.or_missing(hl.is_defined(mt.GT), mt.GT.ploidy))))
mt = mt.annotate_rows(case=group_stats("case"), control=group_stats("control"))
mt = mt.filter_rows((mt.case.call_rate >= MIN_CALL_RATE) & (mt.control.call_rate >= MIN_CALL_RATE))
mt = mt.filter_rows((mt.case.AC + mt.control.AC) > 0)
mt = mt.annotate_rows(af_difference=mt.case.AC/mt.case.AN - mt.control.AC/mt.control.AN)
mt = mt.persist()
assert mt.count_rows() > 0, "No retained variants; inspect interval, schema and QC."


In [ ]:
# Per-person nonreference-site burden; do not display individual samples.
mt = mt.annotate_cols(
    snps=hl.agg.count_where((mt.kind == "SNP") & mt.GT.is_non_ref()),
    indels=hl.agg.count_where((mt.kind == "indel") & mt.GT.is_non_ref()))
samples = mt.cols()
summary = samples.group_by(group=samples.group).aggregate(
    n=hl.agg.count(),
    snps=hl.agg.approx_quantiles(samples.snps, [0.25, 0.5, 0.75]),
    indels=hl.agg.approx_quantiles(samples.indels, [0.25, 0.5, 0.75])).to_pandas()
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, metric in zip(axes, ["snps", "indels"]):
    for i, row in summary.iterrows():
        q1, median, q3 = row[metric]
        ax.errorbar(i, median, yerr=[[median-q1], [q3-median]], fmt="o", capsize=4)
    ax.set(xticks=range(len(summary)), xticklabels=summary.group,
           title=metric.upper() + " nonreference sites", ylabel="Median and IQR per person")
fig.suptitle((INTERVAL or "All autosomal sites in callset") + " — descriptive, unadjusted")
fig.tight_layout()
plt.show()
# Distributions of per-variant AF differences, aggregated in Hail.
rows = mt.rows()
fig, ax = plt.subplots(figsize=(7, 3))
for kind in ["SNP", "indel"]:
    histogram = rows.aggregate(hl.agg.filter(rows.kind == kind,
        hl.agg.hist(rows.af_difference, -1, 1, 40)))
    centers = [(a+b)/2 for a,b in zip(histogram.bin_edges[:-1], histogram.bin_edges[1:])]
    total = sum(histogram.bin_freq)
    if total >= 20:
        ax.plot(centers, [n/total for n in histogram.bin_freq], label=kind)
ax.set(xlabel="Alternate allele frequency: cases minus controls", ylabel="Fraction of retained variants")
ax.legend()
plt.show()
# Keep these outputs in Workbench. Apply current AoU dissemination rules before sharing.


## Interpretation and next steps
All eligible manifest cases in the selected callset are retained, but the manifest can miss undiagnosed or miscoded disease. With INTERVAL=None the notebook scans all autosomal sites in that callset; setting an interval makes this a regional smoke test. Whole-genome scale requires appropriately sized Spark compute and a suitable callset. ACAF/exome/ClinVar callsets have ascertainment restrictions and cannot represent all rare genome-wide variants. Expand only after checking the pilot and compute/storage budget; for the full VDS, subset samples/regions before densifying using the official AoU workflow.

Burden or AF differences can reflect ancestry, age, batch, coverage and ascertainment. A 500:1 ratio may exhaust the eligible pool and does not provide 500 times the information. Interaction testing requires a separate prespecified design, ancestry/relatedness adjustment, multiplicity control and independent validation. Nothing here establishes causal combinations.

Sources: [AoU smaller callsets](https://support.researchallofus.org/hc/en-us/articles/14929793660948-Smaller-Callsets-for-Analyzing-Short-Read-WGS-SNP-Indel-Data-with-Hail-MT-VCF-and-PLINK), [Hail genetics and QC](https://www.hail.is/docs/0.2/methods/genetics.html), [hEDS GeneReviews](https://www.ncbi.nlm.nih.gov/books/NBK1279/).
